### Prediction without Refinement - Soft+Hard Independent Hard-Material Test

Evaluate the soft+hard trained diffusion + forward pipeline on the independent hard-material test set.

#### Inputs
- Test data: `data/HardMaterial_Independent_Test_5Groups_calculated.xlsx`
- Diffusion: `artifacts/high_stiffness/diffusion_model_soft_hard_best.pt` (from `diffusion_model_high_stiffness.ipynb`)
- Forward: `artifacts/high_stiffness/forward_model_weights_soft_hard.pt` (from `forward_model_high_stiffness.ipynb`)
- Scalers: `artifacts/high_stiffness/scaler_x_soft_hard.pkl`, `artifacts/high_stiffness/scaler_y_soft_hard.pkl`

#### Procedure 
1. Use **all 15 independent test rows** (5 groups x 3 samples each)
2. Condition diffusion on 6 moduli + target `(EpH, EppH)` -> sample `(t1, t2)` candidates
3. Forward model: moduli + predicted `(t1, t2)` -> predicted `(EpH, EppH)`
4. Pick best candidate by MSE to target (no refinement)
5. Compare original vs predicted `t1/t2` and target vs predicted `EpH/EppH`
6. Save Excel + candidate scatter plots
7. Section 6: same pipeline on `HardMaterial_Independent_Test_Selected15.xlsx` (15 rows, one plot cell each)

**No retraining / no scaler refitting.**


### Environment Settings

In [ ]:
import math
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib.ticker import MaxNLocator, MultipleLocator

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Serif + bold global plot style (same as previous prediction notebook)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.width'] = 3


### 1. Data and Models Loading

#### 1.1 Forward Model Loading (soft+hard architecture)

In [ ]:
class ImprovedRegressionModel(nn.Module):
    """Match forward_model_high_stiffness.ipynb / train_forward_soft_hard.py"""
    def __init__(self, moduli_dim=6, dist_dim=2, hidden_dim=256,
                 trunk_layers=2, dropout=0.15, trunk_dropout=0.20):
        super().__init__()
        assert dist_dim == 2
        self.moduli_dim = moduli_dim
        self.dist_dim = dist_dim

        trunk = []
        in_dim = moduli_dim
        for _ in range(trunk_layers):
            trunk.append(nn.Linear(in_dim, hidden_dim))
            trunk.append(nn.LayerNorm(hidden_dim))
            trunk.append(nn.GELU())
            trunk.append(nn.Dropout(trunk_dropout))
            in_dim = hidden_dim
        self.trunk = nn.Sequential(*trunk)

        self.dist_embed = nn.Sequential(
            nn.Linear(dist_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        fusion_dim = hidden_dim + (hidden_dim // 4)
        self.fusion_norm = nn.LayerNorm(fusion_dim)

        self.head0 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.head1 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
        )
        self.log_sigma0 = nn.Parameter(torch.zeros(()))
        self.log_sigma1 = nn.Parameter(torch.zeros(()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_mod = x[:, : self.moduli_dim]
        x_dist = x[:, self.moduli_dim :]
        z = self.fusion_norm(torch.cat([self.trunk(x_mod), self.dist_embed(x_dist)], dim=1))
        return torch.cat([self.head0(z), self.head1(z)], dim=-1)


FORWARD_WEIGHTS = Path("artifacts/high_stiffness/forward_model_weights_soft_hard.pt")
if not FORWARD_WEIGHTS.exists():
    raise FileNotFoundError(f"Missing {FORWARD_WEIGHTS}. Train forward_model_high_stiffness.ipynb first.")

forward_model = ImprovedRegressionModel(
    moduli_dim=6, dist_dim=2, hidden_dim=256,
    trunk_layers=2, dropout=0.15, trunk_dropout=0.20,
).to(device)
forward_model.load_state_dict(torch.load(FORWARD_WEIGHTS, map_location=device))
forward_model.eval()
print(f"Loaded forward model from: {FORWARD_WEIGHTS}")


#### 1.2 Diffusion Model Loading (soft+hard)

In [ ]:
class DiffusionSchedule(nn.Module):
    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, eps: float = 1e-8):
        super().__init__()
        self.T = T
        self.eps = eps

        beta = torch.linspace(beta_start, beta_end, T)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]], dim=0)

        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0[:, 6:])

        ab = self.alpha_bar.to(x0.device)[t].clamp(self.eps, 1.0 - self.eps)
        sqrt_ab = ab.sqrt().view(-1, 1)
        sqrt_omb = (1.0 - ab).sqrt().view(-1, 1)

        x_mod = x0[:, :6]  # Ep1-Epp3 remain unchanged
        x_dist = x0[:, 6:]  # t1, t2 undergo diffusion
        x_dist_t = sqrt_ab * x_dist + sqrt_omb * noise

        return torch.cat([x_mod, x_dist_t], dim=1)

print("Diffusion scheduler defined")

class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        half = dim // 2
        factor = math.log(10000.0) / (half - 1)
        self.register_buffer('factor', torch.tensor(factor))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        emb_scale = torch.exp(-self.factor * torch.arange(half, device=t.device))
        emb = t[:, None].float() * emb_scale[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=1)

print("Time embedding defined")

class EnhancedDiffusionModel(nn.Module):
    """Enhanced diffusion model: deeper network with better regularization"""
    def __init__(self, x_dim=8, y_dim=2, time_dim=32, hidden_dim=256):
        super().__init__()
        self.x_dim = x_dim

        # === Moduli branch (Ep1-Epp3): enhanced capacity ===
        self.moduli_embed = nn.Sequential(
            nn.Linear(6, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Distribution branch (t1, t2): deep MLP ===
        self.dist_mlp = nn.Sequential(
            nn.Linear(2, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Y conditioning branch (EpH, EppH) ===
        self.y_mlp = nn.Sequential(
            nn.Linear(y_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128)
        )

        # === Time embedding branch ===
        self.time_embed = TimeEmbedding(time_dim)
        self.time_fc = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

        # === Fusion layer: deep network ===
        fusion_dim = 128 + 128 + 128 + 128  # moduli + dist + y + time
        self.fusion_fc = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)  # Output noise prediction (t1, t2)
        )

    def forward(self, x, t, y):
        """Forward propagation
        x: [B, 8] - [Ep1, Ep2, Ep3, Epp1, Epp2, Epp3, t1_noisy, t2_noisy]
        t: [B] - timestep
        y: [B, 2] - condition [EpH, EppH]
        """
        x_mod = x[:, :6]   # Ep1-Epp3
        x_dist = x[:, 6:]  # t1, t2 (noisy)

        # Feature extraction from each branch
        mod_feat = self.moduli_embed(x_mod)
        dist_feat = self.dist_mlp(x_dist)
        y_feat = self.y_mlp(y)
        t_emb = self.time_embed(t)
        t_feat = self.time_fc(t_emb)

        # Fuse all features
        fused = torch.cat([mod_feat, dist_feat, y_feat, t_feat], dim=1)
        eps_pred = self.fusion_fc(fused)

        return eps_pred

print("Enhanced diffusion model defined")

DIFFUSION_WEIGHTS = Path("artifacts/high_stiffness/diffusion_model_soft_hard_best.pt")
if not DIFFUSION_WEIGHTS.exists():
    raise FileNotFoundError(f"Missing {DIFFUSION_WEIGHTS}. Train diffusion_model_high_stiffness.ipynb first.")

diffusion_model = EnhancedDiffusionModel(x_dim=8, y_dim=2, time_dim=32, hidden_dim=384).to(device)
diffusion_model.load_state_dict(torch.load(DIFFUSION_WEIGHTS, map_location=device))
diffusion_model.eval()
print(f"Loaded diffusion model from: {DIFFUSION_WEIGHTS}")


#### 1.3 Scalers and Independent Test Data

In [ ]:
X_cols = ["Ep1", "Ep2", "Ep3", "Epp1", "Epp2", "Epp3", "t1", "t2"]
y_cols = ["EpH", "EppH"]
MOD = ["Ep1", "Ep2", "Ep3", "Epp1", "Epp2", "Epp3"]

SX_PATH = Path("artifacts/high_stiffness/scaler_x_soft_hard.pkl")
SY_PATH = Path("artifacts/high_stiffness/scaler_y_soft_hard.pkl")
TEST_PATH = Path("data/HardMaterial_Independent_Test_5Groups_calculated.xlsx")

for p in [SX_PATH, SY_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")

scaler_x = joblib.load(SX_PATH)
scaler_y = joblib.load(SY_PATH)
print(f"Loaded scalers:\n  - {SX_PATH}\n  - {SY_PATH}")

df_test_all = pd.read_excel(TEST_PATH, sheet_name="IndependentTestCalculated")
missing = [c for c in X_cols + y_cols + ["IndependentTestGroupID", "SampleIDWithinGroup"] if c not in df_test_all.columns]
if missing:
    raise ValueError(f"Independent test file missing columns: {missing}")

print(f"\nIndependent test full table: {len(df_test_all)} rows, "
      f"{df_test_all['IndependentTestGroupID'].nunique()} groups")
print(df_test_all.groupby("IndependentTestGroupID").size().rename("n_rows").to_string())


#### 1.4 Use All Independent Test Samples

Use the full independent test table: **5 groups x 3 samples = 15 rows**.
No down-selection. Rows are sorted by `IndependentTestGroupID` then `SampleIDWithinGroup` for reproducibility.


In [ ]:
# Use all 15 independent test samples (5 groups x 3 samples)
df_test = (
    df_test_all
    .sort_values(["IndependentTestGroupID", "SampleIDWithinGroup"])
    .reset_index(drop=True)
    .copy()
)

for c in ["IndependentTestGroupID", "SampleIDWithinGroup", "SourceSoftGroupID", "HardSetID"]:
    if c in df_test.columns:
        df_test[c] = df_test[c].astype(int)

print("Using ALL independent test samples (no down-selection).")
print(f"Total rows: {len(df_test)}")
print(f"Groups: {df_test['IndependentTestGroupID'].nunique()}")
print(f"Rows per group:\n{df_test.groupby('IndependentTestGroupID').size().rename('n_rows').to_string()}")

show_cols = [
    "IndependentTestGroupID", "SampleIDWithinGroup",
    "SourceSoftGroupID", "HardSetID",
    "EpH", "EppH", "t1", "t2",
]
print("\nAll 15 test samples:")
print(df_test[show_cols].to_string(index=False))

n_unique_targets = df_test[y_cols].drop_duplicates().shape[0]
print(f"\nUnique (EpH, EppH) targets among all rows: {n_unique_targets}/{len(df_test)}")
assert len(df_test) == 15, f"Expected 15 rows, got {len(df_test)}"
assert df_test["IndependentTestGroupID"].nunique() == 5


#### 1.5 Diffusion Schedule and Sampling

In [ ]:
schedule = DiffusionSchedule(T=1000).to(device)


@torch.no_grad()
def sample_with_fixed_moduli(diffusion_model, schedule, y_cond, moduli_fixed):
    """DDPM reverse sampling; moduli fixed, only (t1, t2) diffused."""
    x_dist = torch.randn(1, 2, device=device)
    x = torch.cat([moduli_fixed, x_dist], dim=1)

    for t in reversed(range(schedule.T)):
        t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
        eps_dist = diffusion_model(x, t_tensor, y_cond)

        beta_t = schedule.beta[t]
        alpha_t = schedule.alpha[t]
        alpha_bar_t = schedule.alpha_bar[t]
        denom_sqrt = torch.sqrt((1 - alpha_bar_t).clamp(min=schedule.eps))

        x_mod_part = moduli_fixed
        x_dist_part = x[:, 6:]

        mu_dist = (1 / torch.sqrt(alpha_t.clamp(min=schedule.eps))) * (
            x_dist_part - (beta_t / denom_sqrt) * eps_dist
        )

        if t > 0:
            ab_prev = schedule.alpha_bar_prev[t]
            sigma_t = torch.sqrt(beta_t * (1 - ab_prev) / (1 - alpha_bar_t))
            z = torch.randn_like(mu_dist)
            x_dist_prev = mu_dist + sigma_t * z
        else:
            x_dist_prev = mu_dist

        x = torch.cat([x_mod_part, x_dist_prev], dim=1)

    return x


print(f"Diffusion schedule ready: T={schedule.T}")


### 2. Independent Hard-Material Predictions

#### 2.1 Prediction helpers

Shared functions for sampling, forward prediction, error metrics, and plotting.
Axis limits use **larger padding** so the legend does not cover the target / best-candidate markers.


In [ ]:
NUM_SAMPLES = 250
REL_EPS = 1e-12  # avoid /0 in relative error

os.makedirs("image", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Accumulate results across group cells (re-run earlier groups if you restart mid-way)
all_results = []

print(f"Ready to process {len(df_test)} independent samples "
      f"({NUM_SAMPLES} diffusion candidates each), group by group.")


def adaptive_axis_limits(x_val, y_val, preds_xy, best_xy=None, pad_frac=0.45):
    """Larger padding so legend / markers are not cramped."""
    pts = [np.asarray([[x_val, y_val]], dtype=float), np.asarray(preds_xy, dtype=float)]
    if best_xy is not None:
        pts.append(np.asarray(best_xy, dtype=float).reshape(1, 2))
    all_xy = np.vstack(pts)
    x_min, x_max = float(all_xy[:, 0].min()), float(all_xy[:, 0].max())
    y_min, y_max = float(all_xy[:, 1].min()), float(all_xy[:, 1].max())
    dx = max(x_max - x_min, abs(x_val) * 0.08 + 1.0)
    dy = max(y_max - y_min, abs(y_val) * 0.08 + 1.0)
    # Extra headroom on top-right (common legend area)
    return (
        x_min - pad_frac * dx,
        x_max + pad_frac * dx * 1.25,
        y_min - pad_frac * dy,
        y_max + pad_frac * dy * 1.25,
    )

def predict_one_sample(row, num_samples=NUM_SAMPLES, show_plots=True):
    """Diffusion + forward prediction for one independent-test row."""
    gid = int(row["IndependentTestGroupID"])
    sid = int(row["SampleIDWithinGroup"])
    src_gid = int(row["SourceSoftGroupID"]) if "SourceSoftGroupID" in row.index else -1
    hard_set = int(row["HardSetID"]) if "HardSetID" in row.index else -1

    fixed_moduli_raw = row[MOD].to_numpy(dtype=float).reshape(1, 6)
    t_orig = row[["t1", "t2"]].to_numpy(dtype=float).reshape(1, 2)
    raw_target = row[y_cols].to_numpy(dtype=float).reshape(1, 2)

    moduli_tmp = np.zeros((1, 8), dtype=float)
    moduli_tmp[0, :6] = fixed_moduli_raw
    fixed_moduli_scaled = scaler_x.transform(moduli_tmp)[:, :6]
    moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)

    std_target = scaler_y.transform(raw_target)
    target_y = torch.tensor(std_target, dtype=torch.float32, device=device)

    print(f"  SourceSoftGroupID={src_gid}, HardSetID={hard_set}")
    print(f"  Fixed moduli: {np.round(fixed_moduli_raw.flatten(), 6)}")
    print(f"  Original t1,t2: {np.round(t_orig.flatten(), 6)}")
    print(f"  Target EpH,EppH: {np.round(raw_target.flatten(), 6)}")

    cands_x_std, preds_std = [], []
    for _ in range(num_samples):
        x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
        cands_x_std.append(x_gen.detach().cpu().numpy().reshape(-1))
        preds_std.append(forward_model(x_gen).detach().cpu().numpy().reshape(-1))

    cands_x_std = np.vstack(cands_x_std)
    preds_std = np.vstack(preds_std)
    cands_x_orig = scaler_x.inverse_transform(cands_x_std)
    preds_orig = scaler_y.inverse_transform(preds_std)

    errors_orig = np.mean((preds_orig - raw_target) ** 2, axis=1)
    best_idx = int(np.argmin(errors_orig))
    best_x = cands_x_orig[best_idx]
    best_y = preds_orig[best_idx]
    best_mse = float(errors_orig[best_idx])

    t1_pred, t2_pred = float(best_x[6]), float(best_x[7])
    eph_pred, epph_pred = float(best_y[0]), float(best_y[1])
    eph_targ, epph_targ = float(raw_target[0, 0]), float(raw_target[0, 1])
    t1_orig, t2_orig = float(t_orig[0, 0]), float(t_orig[0, 1])

    abs_err_eph = abs(eph_pred - eph_targ)
    abs_err_epph = abs(epph_pred - epph_targ)
    rel_err_eph = abs_err_eph / max(abs(eph_targ), REL_EPS)
    rel_err_epph = abs_err_epph / max(abs(epph_targ), REL_EPS)

    print("  Best candidate (no refinement):")
    print(f"    t1_pred,t2_pred = [{t1_pred:.6f}, {t2_pred:.6f}]")
    print(f"    EpH_pred,EppH_pred = [{eph_pred:.6f}, {epph_pred:.6f}]")
    print(f"    MSE(EpH,EppH) = {best_mse:.6e}")
    print(f"    AbsErr EpH={abs_err_eph:.6f}, EppH={abs_err_epph:.6f}")
    print(f"    RelErr EpH={rel_err_eph*100:.3f}%, EppH={rel_err_epph*100:.3f}%")

    if show_plots:
        top_idx = np.argsort(errors_orig)[:50]
        best_xy = np.array([[eph_pred, epph_pred]], dtype=float)
        x_min, x_max, y_min, y_max = adaptive_axis_limits(
            eph_targ, epph_targ, preds_orig[top_idx], best_xy=best_xy, pad_frac=0.45
        )

        plt.figure(figsize=(10, 10))
        plt.scatter(
            preds_orig[top_idx, 0], preds_orig[top_idx, 1],
            c="gray", alpha=0.5, s=50, label="Candidates", zorder=1,
        )
        plt.scatter(
            eph_pred, epph_pred,
            c="blue", s=150, edgecolors="b", label="Best candidate", zorder=3,
        )
        plt.scatter(
            eph_targ, epph_targ,
            c="red", marker="x", s=200,
            label=f"Target ({eph_targ:.3f}, {epph_targ:.3f})", zorder=4,
        )
        plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
        plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)
        plt.xticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.yticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        # Place legend in upper-left; expanded axes keep it off the markers
        plt.legend(
            prop={"size": 18, "weight": "bold"},
            loc="upper left",
            framealpha=0.9,
            borderpad=0.6,
        )
        ax = plt.gca()
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis="both", which="major", length=8, width=3)
        plt.tight_layout()
        large_path = f"image/indep_hard_g{gid}_s{sid}_large.png"
        plt.savefig(large_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {large_path}")

        plt.figure(figsize=(6, 6))
        plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c="gray", alpha=0.5, s=200, zorder=1)
        plt.scatter(eph_pred, epph_pred, c="blue", s=200, edgecolors="k", zorder=3)
        plt.scatter(eph_targ, epph_targ, c="red", marker="x", s=200, zorder=4)
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.xticks([])
        plt.yticks([])
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(4.5)
            spine.set_linestyle("--")
        plt.tight_layout()
        small_path = f"image/indep_hard_g{gid}_s{sid}_small.png"
        plt.savefig(small_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {small_path}")

    result = {
        "IndependentTestGroupID": gid,
        "SampleIDWithinGroup": sid,
        "SourceSoftGroupID": src_gid,
        "HardSetID": hard_set,
        "Ep1": float(fixed_moduli_raw[0, 0]),
        "Ep2": float(fixed_moduli_raw[0, 1]),
        "Ep3": float(fixed_moduli_raw[0, 2]),
        "Epp1": float(fixed_moduli_raw[0, 3]),
        "Epp2": float(fixed_moduli_raw[0, 4]),
        "Epp3": float(fixed_moduli_raw[0, 5]),
        "t1_orig": t1_orig,
        "t2_orig": t2_orig,
        "t1_pred": t1_pred,
        "t2_pred": t2_pred,
        "EpH_targ": eph_targ,
        "EppH_targ": epph_targ,
        "EpH_pred": eph_pred,
        "EppH_pred": epph_pred,
        "AbsErr_EpH": abs_err_eph,
        "AbsErr_EppH": abs_err_epph,
        "RelErr_EpH": rel_err_eph,
        "RelErr_EppH": rel_err_epph,
        "RelErr_EpH_pct": rel_err_eph * 100.0,
        "RelErr_EppH_pct": rel_err_epph * 100.0,
        "BestCandidateMSE": best_mse,
        "NumCandidates": num_samples,
    }
    return result


def run_group(group_id, df_all=None, results_list=None, num_samples=NUM_SAMPLES):
    """Run prediction+plots for one IndependentTestGroupID (its 3 samples)."""
    if df_all is None:
        df_all = df_test
    if results_list is None:
        results_list = all_results

    gdf = (
        df_all.loc[df_all["IndependentTestGroupID"] == group_id]
        .sort_values("SampleIDWithinGroup")
        .reset_index(drop=True)
    )
    print("=" * 70)
    print(f"IndependentTestGroupID = {group_id}  |  {len(gdf)} samples")
    print("=" * 70)

    # Drop previous results for this group so re-running the cell replaces them
    keep = [r for r in results_list if int(r["IndependentTestGroupID"]) != int(group_id)]
    results_list.clear()
    results_list.extend(keep)

    for j, (_, row) in enumerate(gdf.iterrows()):
        sid = int(row["SampleIDWithinGroup"])
        print(f"\n--- Group {group_id} sample {j+1}/{len(gdf)} "
              f"(SampleIDWithinGroup={sid}) ---")
        res = predict_one_sample(row, num_samples=num_samples, show_plots=True)
        results_list.append(res)
        print("-" * 60)

    print(f"\n[Group {group_id}] done. all_results now has {len(results_list)} rows.")
    return results_list


print("Helpers defined: predict_one_sample(), run_group()")


#### 2.2 IndependentTestGroupID = 1

Run **only this cell** to (re)generate predictions and figures for group 1
(3 samples). Other groups are unchanged.


In [ ]:
# Predict & plot IndependentTestGroupID = 1 only
run_group(1)


#### 2.3 IndependentTestGroupID = 2

Run **only this cell** to (re)generate predictions and figures for group 2
(3 samples). Other groups are unchanged.


In [ ]:
# Predict & plot IndependentTestGroupID = 2 only
run_group(2)


#### 2.4 IndependentTestGroupID = 3

Run **only this cell** to (re)generate predictions and figures for group 3
(3 samples). Other groups are unchanged.


In [ ]:
# Predict & plot IndependentTestGroupID = 3 only
run_group(3)


#### 2.5 IndependentTestGroupID = 4

Run **only this cell** to (re)generate predictions and figures for group 4
(3 samples). Other groups are unchanged.


In [ ]:
# Predict & plot IndependentTestGroupID = 4 only
run_group(4)


#### 2.6 IndependentTestGroupID = 5

Run **only this cell** to (re)generate predictions and figures for group 5
(3 samples). Other groups are unchanged.


In [ ]:
# Predict & plot IndependentTestGroupID = 5 only
run_group(5)


### 3. Save Excel and Summary

Run after all group cells you need. Builds the Excel from `all_results`.


In [ ]:
results_df = pd.DataFrame(all_results).sort_values(
    ["IndependentTestGroupID", "SampleIDWithinGroup"]
).reset_index(drop=True)

OUT_XLSX = "data/independent_hard_test_prediction_soft_hard.xlsx"
results_df.to_excel(OUT_XLSX, index=False)

print(f"Results saved to {OUT_XLSX}")
print(f"Total samples in all_results: {len(results_df)}")
print("\nSummary table:")
print(results_df[[
    "IndependentTestGroupID", "SampleIDWithinGroup",
    "t1_orig", "t1_pred", "t2_orig", "t2_pred",
    "EpH_targ", "EpH_pred", "EppH_targ", "EppH_pred",
    "AbsErr_EpH", "RelErr_EpH_pct", "AbsErr_EppH", "RelErr_EppH_pct",
]].round(4).to_string(index=False))

print("\n" + "=" * 70)
print("Independent hard-material prediction summary (soft+hard models)")
print("=" * 70)
print(f"Test file: {TEST_PATH}")
print("Selection: ALL independent test rows, processed group-by-group")
print(f"Diffusion: {DIFFUSION_WEIGHTS}")
print(f"Forward:   {FORWARD_WEIGHTS}")
print(f"Scalers:   {SX_PATH}, {SY_PATH}")
print(f"Candidates per sample: {NUM_SAMPLES}")
print(f"Output Excel: {OUT_XLSX}")
if len(results_df):
    print()
    print(f"Mean AbsErr EpH:  {results_df['AbsErr_EpH'].mean():.6f}")
    print(f"Mean AbsErr EppH: {results_df['AbsErr_EppH'].mean():.6f}")
    print(f"Mean RelErr EpH:  {results_df['RelErr_EpH_pct'].mean():.3f}%")
    print(f"Mean RelErr EppH: {results_df['RelErr_EppH_pct'].mean():.3f}%")
print("=" * 70)


### 4. New Samples Prediction (`data/new samples.xlsx`)

5 validation groups (1 row each) from `data/new samples.xlsx`.

For each group:
1. Condition diffusion on 6 moduli + experimental `(EpH, EppH)`
2. Sample candidates of `(t1, t2)` and pick best via forward-model MSE
3. Compare original vs predicted `t1/t2` and experimental vs predicted `EpH/EppH`
4. Plot large/small scatter figures (same style as Section 2)

**One code cell per group** — re-run only the group you need.
Final Excel: `data/new samples_real_predicted_comparison.xlsx`


#### 4.1 Load new samples and define helpers


In [ ]:
NEW_SAMPLES_PATH = Path("data/new samples.xlsx")
NEW_SAMPLES_OUT = Path("data/new samples_real_predicted_comparison.xlsx")

if not NEW_SAMPLES_PATH.exists():
    raise FileNotFoundError(f"Missing {NEW_SAMPLES_PATH}")

df_new_all = pd.read_excel(NEW_SAMPLES_PATH, header=0)
missing_new = [c for c in X_cols + y_cols if c not in df_new_all.columns]
if missing_new:
    raise ValueError(f"new samples.xlsx missing columns: {missing_new}")

# 5 rows = 5 groups (GroupID 1..5), sorted by original file order
df_new = df_new_all.copy().reset_index(drop=True)
df_new["NewSampleGroupID"] = np.arange(1, len(df_new) + 1, dtype=int)

print(f"Loaded {NEW_SAMPLES_PATH}: {len(df_new)} groups")
print(df_new[["NewSampleGroupID"] + X_cols + y_cols].round(6).to_string(index=False))
assert len(df_new) == 5, f"Expected 5 groups, got {len(df_new)}"

# Separate result list so Section 2 independent-test results stay intact
new_sample_results = []

def predict_new_sample_row(row, num_samples=NUM_SAMPLES, show_plots=True):
    """Same diffusion+forward pipeline as independent test, for one new-sample group."""
    gid = int(row["NewSampleGroupID"])

    fixed_moduli_raw = row[MOD].to_numpy(dtype=float).reshape(1, 6)
    t_orig = row[["t1", "t2"]].to_numpy(dtype=float).reshape(1, 2)
    raw_target = row[y_cols].to_numpy(dtype=float).reshape(1, 2)  # experimental EpH, EppH

    moduli_tmp = np.zeros((1, 8), dtype=float)
    moduli_tmp[0, :6] = fixed_moduli_raw
    fixed_moduli_scaled = scaler_x.transform(moduli_tmp)[:, :6]
    moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)

    std_target = scaler_y.transform(raw_target)
    target_y = torch.tensor(std_target, dtype=torch.float32, device=device)

    print(f"  Fixed moduli: {np.round(fixed_moduli_raw.flatten(), 6)}")
    print(f"  Original t1,t2 (from Excel): {np.round(t_orig.flatten(), 6)}")
    print(f"  Experimental EpH,EppH: {np.round(raw_target.flatten(), 6)}")

    cands_x_std, preds_std = [], []
    for _ in range(num_samples):
        x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
        cands_x_std.append(x_gen.detach().cpu().numpy().reshape(-1))
        preds_std.append(forward_model(x_gen).detach().cpu().numpy().reshape(-1))

    cands_x_std = np.vstack(cands_x_std)
    preds_std = np.vstack(preds_std)
    cands_x_orig = scaler_x.inverse_transform(cands_x_std)
    preds_orig = scaler_y.inverse_transform(preds_std)

    errors_orig = np.mean((preds_orig - raw_target) ** 2, axis=1)
    best_idx = int(np.argmin(errors_orig))
    best_x = cands_x_orig[best_idx]
    best_y = preds_orig[best_idx]
    best_mse = float(errors_orig[best_idx])

    t1_pred, t2_pred = float(best_x[6]), float(best_x[7])
    eph_pred, epph_pred = float(best_y[0]), float(best_y[1])
    eph_targ, epph_targ = float(raw_target[0, 0]), float(raw_target[0, 1])
    t1_orig, t2_orig = float(t_orig[0, 0]), float(t_orig[0, 1])

    abs_err_eph = abs(eph_pred - eph_targ)
    abs_err_epph = abs(epph_pred - epph_targ)
    rel_err_eph = abs_err_eph / max(abs(eph_targ), REL_EPS)
    rel_err_epph = abs_err_epph / max(abs(epph_targ), REL_EPS)

    print("  Best candidate (no refinement):")
    print(f"    t1_pred,t2_pred = [{t1_pred:.6f}, {t2_pred:.6f}]")
    print(f"    EpH_pred,EppH_pred = [{eph_pred:.6f}, {epph_pred:.6f}]")
    print(f"    MSE(EpH,EppH) = {best_mse:.6e}")
    print(f"    AbsErr EpH={abs_err_eph:.6f}, EppH={abs_err_epph:.6f}")
    print(f"    RelErr EpH={rel_err_eph*100:.3f}%, EppH={rel_err_epph*100:.3f}%")

    if show_plots:
        top_idx = np.argsort(errors_orig)[:50]
        best_xy = np.array([[eph_pred, epph_pred]], dtype=float)
        x_min, x_max, y_min, y_max = adaptive_axis_limits(
            eph_targ, epph_targ, preds_orig[top_idx], best_xy=best_xy, pad_frac=0.45
        )

        plt.figure(figsize=(10, 10))
        plt.scatter(
            preds_orig[top_idx, 0], preds_orig[top_idx, 1],
            c="gray", alpha=0.5, s=50, label="Candidates", zorder=1,
        )
        plt.scatter(
            eph_pred, epph_pred,
            c="blue", s=150, edgecolors="b", label="Best candidate", zorder=3,
        )
        plt.scatter(
            eph_targ, epph_targ,
            c="red", marker="x", s=200,
            label=f"Target ({eph_targ:.3f}, {epph_targ:.3f})", zorder=4,
        )
        plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
        plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)
        plt.xticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.yticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.legend(prop={"size": 18, "weight": "bold"}, loc="upper left", framealpha=0.9, borderpad=0.6)
        ax = plt.gca()
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis="both", which="major", length=8, width=3)
        plt.tight_layout()
        large_path = f"image/new_sample_g{gid}_large.png"
        plt.savefig(large_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {large_path}")

        plt.figure(figsize=(6, 6))
        plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c="gray", alpha=0.5, s=200, zorder=1)
        plt.scatter(eph_pred, epph_pred, c="blue", s=200, edgecolors="k", zorder=3)
        plt.scatter(eph_targ, epph_targ, c="red", marker="x", s=200, zorder=4)
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.xticks([])
        plt.yticks([])
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(4.5)
            spine.set_linestyle("--")
        plt.tight_layout()
        small_path = f"image/new_sample_g{gid}_small.png"
        plt.savefig(small_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {small_path}")

    return {
        "NewSampleGroupID": gid,
        "Ep1": float(fixed_moduli_raw[0, 0]),
        "Ep2": float(fixed_moduli_raw[0, 1]),
        "Ep3": float(fixed_moduli_raw[0, 2]),
        "Epp1": float(fixed_moduli_raw[0, 3]),
        "Epp2": float(fixed_moduli_raw[0, 4]),
        "Epp3": float(fixed_moduli_raw[0, 5]),
        "t1_orig": t1_orig,
        "t2_orig": t2_orig,
        "t1_pred": t1_pred,
        "t2_pred": t2_pred,
        "EpH_real": eph_targ,
        "EppH_real": epph_targ,
        "EpH_pred": eph_pred,
        "EppH_pred": epph_pred,
        "AbsErr_EpH": abs_err_eph,
        "AbsErr_EppH": abs_err_epph,
        "RelErr_EpH": rel_err_eph,
        "RelErr_EppH": rel_err_epph,
        "RelErr_EpH_pct": rel_err_eph * 100.0,
        "RelErr_EppH_pct": rel_err_epph * 100.0,
        "BestCandidateMSE": best_mse,
        "NumCandidates": num_samples,
    }


def run_new_sample_group(group_id, df_all=None, results_list=None, num_samples=NUM_SAMPLES):
    """Predict + plot one new-sample group (1 row). Re-running replaces that group's result."""
    if df_all is None:
        df_all = df_new
    if results_list is None:
        results_list = new_sample_results

    gdf = df_all.loc[df_all["NewSampleGroupID"] == group_id].reset_index(drop=True)
    if len(gdf) != 1:
        raise ValueError(f"Expected 1 row for NewSampleGroupID={group_id}, got {len(gdf)}")

    print("=" * 70)
    print(f"NewSampleGroupID = {group_id}")
    print("=" * 70)

    keep = [r for r in results_list if int(r["NewSampleGroupID"]) != int(group_id)]
    results_list.clear()
    results_list.extend(keep)

    row = gdf.iloc[0]
    res = predict_new_sample_row(row, num_samples=num_samples, show_plots=True)
    results_list.append(res)
    print(f"\n[NewSampleGroup {group_id}] done. new_sample_results has {len(results_list)} rows.")
    print("-" * 60)
    return results_list


print("New-sample helpers ready: predict_new_sample_row(), run_new_sample_group()")


#### 4.2 NewSampleGroupID = 1

Run **only this cell** to (re)generate prediction and figures for new-sample group 1.


In [ ]:
# Predict & plot new-sample group 1
run_new_sample_group(1)


#### 4.3 NewSampleGroupID = 2

Run **only this cell** to (re)generate prediction and figures for new-sample group 2.


In [ ]:
# Predict & plot new-sample group 2
run_new_sample_group(2)


#### 4.4 NewSampleGroupID = 3

Run **only this cell** to (re)generate prediction and figures for new-sample group 3.


In [ ]:
# Predict & plot new-sample group 3
run_new_sample_group(3)


#### 4.5 NewSampleGroupID = 4

Run **only this cell** to (re)generate prediction and figures for new-sample group 4.


In [ ]:
# Predict & plot new-sample group 4
run_new_sample_group(4)


#### 4.6 NewSampleGroupID = 5

Run **only this cell** to (re)generate prediction and figures for new-sample group 5.


In [ ]:
# Predict & plot new-sample group 5
run_new_sample_group(5)


#### 4.7 Save new-sample comparison Excel

Run after the group cells you need. Writes `data/new samples_real_predicted_comparison.xlsx`.


In [ ]:
new_results_df = pd.DataFrame(new_sample_results).sort_values(
    ["NewSampleGroupID"]
).reset_index(drop=True)

new_results_df.to_excel(NEW_SAMPLES_OUT, index=False)

print(f"Results saved to {NEW_SAMPLES_OUT}")
print(f"Total groups: {len(new_results_df)}")
print("\nReal vs predicted comparison:")
print(new_results_df[[
    "NewSampleGroupID",
    "t1_orig", "t1_pred", "t2_orig", "t2_pred",
    "EpH_real", "EpH_pred", "EppH_real", "EppH_pred",
    "AbsErr_EpH", "RelErr_EpH_pct", "AbsErr_EppH", "RelErr_EppH_pct",
]].round(4).to_string(index=False))

print("\n" + "=" * 70)
print("New samples prediction summary")
print("=" * 70)
print(f"Input:  {NEW_SAMPLES_PATH}")
print(f"Output: {NEW_SAMPLES_OUT}")
print(f"Candidates per group: {NUM_SAMPLES}")
if len(new_results_df):
    print(f"Mean AbsErr EpH:  {new_results_df['AbsErr_EpH'].mean():.6f}")
    print(f"Mean AbsErr EppH: {new_results_df['AbsErr_EppH'].mean():.6f}")
    print(f"Mean RelErr EpH:  {new_results_df['RelErr_EpH_pct'].mean():.3f}%")
    print(f"Mean RelErr EppH: {new_results_df['RelErr_EppH_pct'].mean():.3f}%")
print("=" * 70)


### 5. Gradient Refinement on New Samples

Apply the same **gradient refinement** procedure as Section 2 of `prediction_with_refinement.ipynb`
to the 5 groups in `data/new samples.xlsx`.

#### Method
1. Diffusion sample 250 candidates → pick best by MSE to experimental `(EpH, EppH)`
2. Fix 6 moduli; optimize only standardized `(t1, t2)` with Adam (`lr=1e-2`, 100 steps)
3. Loss = MSE between forward prediction and target in **standardized y-space**
4. Plot candidates / before / after / target  
   - **After refinement**: green point  
   - **Target**: red cross (`x`)

One cell per group. Final Excel: `data/new samples_refined_comparison.xlsx`


#### 5.1 Refinement helpers


In [ ]:
REFINE_LR = 1e-2
REFINE_STEPS = 100
REFINE_SNAP_EVERY = 10
NEW_SAMPLES_REFINED_OUT = Path("data/new samples_refined_comparison.xlsx")

# Keep Section-4 new-sample table if already loaded; otherwise reload
if "df_new" not in globals():
    NEW_SAMPLES_PATH = Path("data/new samples.xlsx")
    df_new = pd.read_excel(NEW_SAMPLES_PATH, header=0).copy().reset_index(drop=True)
    df_new["NewSampleGroupID"] = np.arange(1, len(df_new) + 1, dtype=int)
    print(f"Reloaded {NEW_SAMPLES_PATH}: {len(df_new)} groups")

new_refined_results = []


def refine_dist_from_best(
    moduli_fixed,
    cands_x_std,
    preds_orig,
    raw_target,
    lr=REFINE_LR,
    num_steps=REFINE_STEPS,
    snap_every=REFINE_SNAP_EVERY,
):
    """Section-2 style refinement: Adam on std-space (t1, t2) only; moduli fixed."""
    errors_orig = np.mean((preds_orig - raw_target) ** 2, axis=1)
    best_idx = int(np.argmin(errors_orig))

    dist_var = torch.tensor(
        cands_x_std[best_idx:best_idx + 1, 6:],
        dtype=torch.float32,
        device=device,
        requires_grad=True,
    )
    opt = torch.optim.Adam([dist_var], lr=lr)
    std_tgt = torch.tensor(
        scaler_y.transform(raw_target), dtype=torch.float32, device=device
    )

    snap_steps = list(range(0, num_steps, snap_every))
    snapshot_preds = []

    # Keep model in eval() (same as refinement notebook); gradients still flow to dist_var
    forward_model.eval()
    for step in range(num_steps):
        opt.zero_grad()
        y_hat = forward_model(torch.cat([moduli_fixed, dist_var], dim=1))
        loss = nn.MSELoss()(y_hat, std_tgt)
        loss.backward()
        opt.step()
        if step in snap_steps:
            with torch.no_grad():
                snapshot_preds.append(
                    forward_model(torch.cat([moduli_fixed, dist_var], dim=1))
                    .detach().cpu().numpy().flatten()
                )

    refined_x_std = np.hstack([
        moduli_fixed.detach().cpu().numpy(),
        dist_var.detach().cpu().numpy(),
    ])
    with torch.no_grad():
        refined_y_std = forward_model(
            torch.tensor(refined_x_std, dtype=torch.float32, device=device)
        ).detach().cpu().numpy()

    refined_x_orig = scaler_x.inverse_transform(refined_x_std)
    refined_y_orig = scaler_y.inverse_transform(refined_y_std)
    snapshot_preds_orig = scaler_y.inverse_transform(np.asarray(snapshot_preds, dtype=float))
    cands_x_orig = scaler_x.inverse_transform(cands_x_std)

    return {
        "best_idx": best_idx,
        "errors_orig": errors_orig,
        "cands_x_orig": cands_x_orig,
        "orig_x_orig": cands_x_orig[best_idx],
        "orig_y_orig": preds_orig[best_idx],
        "refined_x_orig": refined_x_orig,
        "refined_y_orig": refined_y_orig,
        "refined_mse": float(np.mean((refined_y_orig - raw_target) ** 2)),
        "snapshot_preds_orig": snapshot_preds_orig,
    }


def plot_refinement_result(gid, raw_target, preds_orig, ref):
    """Large/small scatter + iteration path. After=green point, Target=red cross."""
    top_idx = np.argsort(ref["errors_orig"])[:50]
    before_y = ref["orig_y_orig"]
    after_y = ref["refined_y_orig"].flatten()
    eph_targ, epph_targ = float(raw_target[0, 0]), float(raw_target[0, 1])

    pts = np.vstack([
        preds_orig[top_idx],
        before_y.reshape(1, 2),
        after_y.reshape(1, 2),
        raw_target.reshape(1, 2),
    ])
    x_min, x_max, y_min, y_max = adaptive_axis_limits(
        eph_targ, epph_targ, pts, best_xy=after_y.reshape(1, 2), pad_frac=0.45
    )

    # ---- Large comparison plot ----
    plt.figure(figsize=(10, 10))
    plt.scatter(
        preds_orig[top_idx, 0], preds_orig[top_idx, 1],
        c="gray", alpha=0.5, s=50, label="Candidates", zorder=1,
    )
    plt.scatter(
        before_y[0], before_y[1],
        c="blue", s=150, edgecolors="b", label="Before refinement", zorder=3,
    )
    plt.scatter(
        after_y[0], after_y[1],
        c="green", s=180, edgecolors="darkgreen", linewidths=1.5,
        label="After refinement", zorder=4,
    )
    plt.scatter(
        eph_targ, epph_targ,
        c="red", marker="x", s=220, linewidths=3,
        label=f"Target ({eph_targ:.3f}, {epph_targ:.3f})", zorder=5,
    )
    plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
    plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)
    plt.xticks(fontsize=22, fontweight="bold", fontfamily="serif")
    plt.yticks(fontsize=22, fontweight="bold", fontfamily="serif")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.legend(prop={"size": 16, "weight": "bold"}, loc="upper left", framealpha=0.9)
    ax = plt.gca()
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    plt.tick_params(axis="both", which="major", length=8, width=3)
    plt.tight_layout()
    large_path = f"image/new_sample_g{gid}_refined_large.png"
    plt.savefig(large_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved {large_path}")

    # ---- Small plot ----
    plt.figure(figsize=(6, 6))
    plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c="gray", alpha=0.5, s=200, zorder=1)
    plt.scatter(before_y[0], before_y[1], c="blue", s=200, edgecolors="k", zorder=3)
    plt.scatter(after_y[0], after_y[1], c="green", s=220, edgecolors="k", zorder=4)
    plt.scatter(eph_targ, epph_targ, c="red", marker="x", s=220, linewidths=3, zorder=5)
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xticks([])
    plt.yticks([])
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle("--")
    plt.tight_layout()
    small_path = f"image/new_sample_g{gid}_refined_small.png"
    plt.savefig(small_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved {small_path}")

    # ---- Iteration path (After=green point, Target=red cross) ----
    sp = ref["snapshot_preds_orig"]
    n = len(sp)
    plt.figure(figsize=(10, 10))
    plt.plot(
        sp[:, 0], sp[:, 1], "--o", markevery=range(max(n - 1, 1)),
        label="Iteration Path", linewidth=3, markersize=8, markeredgewidth=2,
        alpha=0.6, color="red",
    )
    plt.scatter(
        sp[-1, 0], sp[-1, 1],
        s=200, edgecolor="black", facecolor="green", linewidth=2,
        label="After refinement", zorder=4,
    )
    plt.scatter(
        [eph_targ], [epph_targ],
        c="red", marker="x", s=300, linewidths=3,
        label=f"Target ({eph_targ:.3f}, {epph_targ:.3f})", zorder=5,
    )
    plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
    plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)
    plt.xticks(fontsize=22, fontweight="bold", fontfamily="serif")
    plt.yticks(fontsize=22, fontweight="bold", fontfamily="serif")
    plt.legend(prop={"size": 16, "weight": "bold"}, loc="upper left", framealpha=0.9)
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    plt.tick_params(axis="both", which="major", length=8, width=3)
    plt.tight_layout()
    path_png = f"image/new_sample_g{gid}_refined_path.png"
    plt.savefig(path_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved {path_png}")


def predict_new_sample_with_refinement(row, num_samples=NUM_SAMPLES, show_plots=True):
    gid = int(row["NewSampleGroupID"])
    fixed_moduli_raw = row[MOD].to_numpy(dtype=float).reshape(1, 6)
    t_orig = row[["t1", "t2"]].to_numpy(dtype=float).reshape(1, 2)
    raw_target = row[y_cols].to_numpy(dtype=float).reshape(1, 2)

    moduli_tmp = np.zeros((1, 8), dtype=float)
    moduli_tmp[0, :6] = fixed_moduli_raw
    moduli_fixed = torch.tensor(
        scaler_x.transform(moduli_tmp)[:, :6], dtype=torch.float32, device=device
    )
    target_y = torch.tensor(
        scaler_y.transform(raw_target), dtype=torch.float32, device=device
    )

    print(f"  Fixed moduli: {np.round(fixed_moduli_raw.flatten(), 6)}")
    print(f"  Original t1,t2 (Excel): {np.round(t_orig.flatten(), 6)}")
    print(f"  Experimental EpH,EppH: {np.round(raw_target.flatten(), 6)}")

    cands_x_std, preds_std = [], []
    forward_model.eval()
    for _ in range(num_samples):
        x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
        cands_x_std.append(x_gen.detach().cpu().numpy().reshape(-1))
        preds_std.append(forward_model(x_gen).detach().cpu().numpy().reshape(-1))
    cands_x_std = np.vstack(cands_x_std)
    preds_std = np.vstack(preds_std)
    preds_orig = scaler_y.inverse_transform(preds_std)

    ref = refine_dist_from_best(moduli_fixed, cands_x_std, preds_orig, raw_target)
    rx = ref["refined_x_orig"].flatten()
    ry = ref["refined_y_orig"].flatten()
    before_mse = float(ref["errors_orig"][ref["best_idx"]])

    print("  Before refinement:")
    print(f"    t1,t2 = [{ref['orig_x_orig'][6]:.6f}, {ref['orig_x_orig'][7]:.6f}]")
    print(f"    EpH,EppH = [{ref['orig_y_orig'][0]:.6f}, {ref['orig_y_orig'][1]:.6f}]")
    print(f"    MSE = {before_mse:.6e}")
    print("  After refinement:")
    print(f"    t1,t2 = [{rx[6]:.6f}, {rx[7]:.6f}]")
    print(f"    EpH,EppH = [{ry[0]:.6f}, {ry[1]:.6f}]")
    print(f"    MSE = {ref['refined_mse']:.6e}")

    if show_plots:
        plot_refinement_result(gid, raw_target, preds_orig, ref)

    eph_real, epph_real = float(raw_target[0, 0]), float(raw_target[0, 1])
    abs_err_eph = abs(ry[0] - eph_real)
    abs_err_epph = abs(ry[1] - epph_real)
    rel_err_eph = abs_err_eph / max(abs(eph_real), REL_EPS)
    rel_err_epph = abs_err_epph / max(abs(epph_real), REL_EPS)

    return {
        "NewSampleGroupID": gid,
        "Ep1": float(fixed_moduli_raw[0, 0]),
        "Ep2": float(fixed_moduli_raw[0, 1]),
        "Ep3": float(fixed_moduli_raw[0, 2]),
        "Epp1": float(fixed_moduli_raw[0, 3]),
        "Epp2": float(fixed_moduli_raw[0, 4]),
        "Epp3": float(fixed_moduli_raw[0, 5]),
        "t1_orig": float(t_orig[0, 0]),
        "t2_orig": float(t_orig[0, 1]),
        "t1_pred_before": float(ref["orig_x_orig"][6]),
        "t2_pred_before": float(ref["orig_x_orig"][7]),
        "t1_pred_after": float(rx[6]),
        "t2_pred_after": float(rx[7]),
        "EpH_real": eph_real,
        "EppH_real": epph_real,
        "EpH_pred_before": float(ref["orig_y_orig"][0]),
        "EppH_pred_before": float(ref["orig_y_orig"][1]),
        "EpH_pred_after": float(ry[0]),
        "EppH_pred_after": float(ry[1]),
        "AbsErr_EpH_after": abs_err_eph,
        "AbsErr_EppH_after": abs_err_epph,
        "RelErr_EpH_pct_after": rel_err_eph * 100.0,
        "RelErr_EppH_pct_after": rel_err_epph * 100.0,
        "BestCandidateMSE": before_mse,
        "RefinedMSE": ref["refined_mse"],
        "NumCandidates": num_samples,
        "RefineLR": REFINE_LR,
        "RefineSteps": REFINE_STEPS,
    }


def run_new_sample_refinement_group(group_id, df_all=None, results_list=None, num_samples=NUM_SAMPLES):
    if df_all is None:
        df_all = df_new
    if results_list is None:
        results_list = new_refined_results

    gdf = df_all.loc[df_all["NewSampleGroupID"] == group_id].reset_index(drop=True)
    if len(gdf) != 1:
        raise ValueError(f"Expected 1 row for NewSampleGroupID={group_id}, got {len(gdf)}")

    print("=" * 70)
    print(f"Refinement | NewSampleGroupID = {group_id}")
    print("=" * 70)

    keep = [r for r in results_list if int(r["NewSampleGroupID"]) != int(group_id)]
    results_list.clear()
    results_list.extend(keep)

    res = predict_new_sample_with_refinement(gdf.iloc[0], num_samples=num_samples, show_plots=True)
    results_list.append(res)
    print(f"\n[Refined Group {group_id}] done. new_refined_results has {len(results_list)} rows.")
    print("-" * 60)
    return results_list


print("Refinement helpers ready:")
print("  refine_dist_from_best(), predict_new_sample_with_refinement(), run_new_sample_refinement_group()")
print(f"  Adam lr={REFINE_LR}, steps={REFINE_STEPS}, snap every {REFINE_SNAP_EVERY}")


#### 5.2 Refine NewSampleGroupID = 1

Run **only this cell** to (re)generate diffusion + gradient refinement and figures for group 1.


In [ ]:
# Diffusion best candidate + gradient refinement for group 1
run_new_sample_refinement_group(1)


#### 5.3 Refine NewSampleGroupID = 2

Run **only this cell** to (re)generate diffusion + gradient refinement and figures for group 2.


In [ ]:
# Diffusion best candidate + gradient refinement for group 2
run_new_sample_refinement_group(2)


#### 5.4 Refine NewSampleGroupID = 3

Run **only this cell** to (re)generate diffusion + gradient refinement and figures for group 3.


In [ ]:
# Diffusion best candidate + gradient refinement for group 3
run_new_sample_refinement_group(3)


#### 5.5 Refine NewSampleGroupID = 4

Run **only this cell** to (re)generate diffusion + gradient refinement and figures for group 4.


In [ ]:
# Diffusion best candidate + gradient refinement for group 4
run_new_sample_refinement_group(4)


#### 5.6 Refine NewSampleGroupID = 5

Run **only this cell** to (re)generate diffusion + gradient refinement and figures for group 5.


In [ ]:
# Diffusion best candidate + gradient refinement for group 5
run_new_sample_refinement_group(5)


#### 5.7 Save refined comparison Excel

Writes `data/new samples_refined_comparison.xlsx`.


In [ ]:
refined_df = pd.DataFrame(new_refined_results).sort_values(
    ["NewSampleGroupID"]
).reset_index(drop=True)

refined_df.to_excel(NEW_SAMPLES_REFINED_OUT, index=False)

print(f"Results saved to {NEW_SAMPLES_REFINED_OUT}")
print(f"Total groups: {len(refined_df)}")
print("\nBefore vs after refinement:")
print(refined_df[[
    "NewSampleGroupID",
    "t1_pred_before", "t1_pred_after", "t2_pred_before", "t2_pred_after",
    "EpH_real", "EpH_pred_before", "EpH_pred_after",
    "EppH_real", "EppH_pred_before", "EppH_pred_after",
    "BestCandidateMSE", "RefinedMSE",
    "AbsErr_EpH_after", "RelErr_EpH_pct_after",
    "AbsErr_EppH_after", "RelErr_EppH_pct_after",
]].round(4).to_string(index=False))

print("\n" + "=" * 70)
print("New samples + gradient refinement summary")
print("=" * 70)
print(f"Input:  data/new samples.xlsx")
print(f"Output: {NEW_SAMPLES_REFINED_OUT}")
print(f"Candidates/group: {NUM_SAMPLES} | Adam lr={REFINE_LR} | steps={REFINE_STEPS}")
if len(refined_df):
    print(f"Mean BestCandidateMSE: {refined_df['BestCandidateMSE'].mean():.6e}")
    print(f"Mean RefinedMSE:       {refined_df['RefinedMSE'].mean():.6e}")
    print(f"Mean AbsErr EpH after:  {refined_df['AbsErr_EpH_after'].mean():.6f}")
    print(f"Mean AbsErr EppH after: {refined_df['AbsErr_EppH_after'].mean():.6f}")
print("=" * 70)


### 6. Selected15 Independent Hard-Material Predictions

Load `data/HardMaterial_Independent_Test_Selected15.xlsx` (sheet `Selected15`, **15 rows**).

For each row:
1. Condition diffusion on 6 moduli + target `(EpH, EppH)`
2. Sample candidates of `(t1, t2)` and pick best via forward-model MSE (no refinement)
3. Compare original vs predicted `t1/t2` and target vs predicted `EpH/EppH`
4. Plot large/small scatter figures (same style as Section 2 / 4)

**One code cell per row** — re-run only the row you need.
Final Excel: `data/independent_hard_selected15_prediction_soft_hard.xlsx`


#### 6.1 Load Selected15 data and define helpers


In [ ]:
# ---- Section-6 bootstrap (safe if Section 2 helpers were not re-run) ----
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 250
if "REL_EPS" not in globals():
    REL_EPS = 1e-12
os.makedirs("image", exist_ok=True)
os.makedirs("data", exist_ok=True)

if "adaptive_axis_limits" not in globals():
    def adaptive_axis_limits(x_val, y_val, preds_xy, best_xy=None, pad_frac=0.45):
        """Larger padding so legend / markers are not cramped."""
        pts = [np.asarray([[x_val, y_val]], dtype=float), np.asarray(preds_xy, dtype=float)]
        if best_xy is not None:
            pts.append(np.asarray(best_xy, dtype=float).reshape(1, 2))
        all_xy = np.vstack(pts)
        x_min, x_max = float(all_xy[:, 0].min()), float(all_xy[:, 0].max())
        y_min, y_max = float(all_xy[:, 1].min()), float(all_xy[:, 1].max())
        dx = max(x_max - x_min, abs(x_val) * 0.08 + 1.0)
        dy = max(y_max - y_min, abs(y_val) * 0.08 + 1.0)
        return (
            x_min - pad_frac * dx,
            x_max + pad_frac * dx * 1.25,
            y_min - pad_frac * dy,
            y_max + pad_frac * dy * 1.25,
        )

SELECTED15_PATH = Path("data/HardMaterial_Independent_Test_Selected15.xlsx")
SELECTED15_OUT = Path("data/independent_hard_selected15_prediction_soft_hard.xlsx")
SELECTED15_SHEET = "Selected15"

if not SELECTED15_PATH.exists():
    raise FileNotFoundError(f"Missing {SELECTED15_PATH}")

df_sel15_all = pd.read_excel(SELECTED15_PATH, sheet_name=SELECTED15_SHEET)
missing_sel = [c for c in X_cols + y_cols if c not in df_sel15_all.columns]
if missing_sel:
    raise ValueError(f"{SELECTED15_PATH.name} missing columns: {missing_sel}")

# Keep Excel row order; assign SelectedRowID = 1..15 for one-cell-per-row runs
df_sel15 = df_sel15_all.copy().reset_index(drop=True)
df_sel15["SelectedRowID"] = np.arange(1, len(df_sel15) + 1, dtype=int)

for c in ["IndependentTestGroupID", "SampleIDWithinGroup", "SourceSoftGroupID", "HardSetID"]:
    if c in df_sel15.columns:
        df_sel15[c] = df_sel15[c].astype(int)

print(f"Loaded {SELECTED15_PATH} [{SELECTED15_SHEET}]: {len(df_sel15)} rows")
show_cols = [
    "SelectedRowID",
    "IndependentTestGroupID", "SampleIDWithinGroup",
    "SourceSoftGroupID", "HardSetID",
    "EpH", "EppH", "t1", "t2",
]
print(df_sel15[show_cols].round(6).to_string(index=False))
assert len(df_sel15) == 15, f"Expected 15 rows, got {len(df_sel15)}"

# Separate result list so earlier sections stay intact
selected15_results = []

def predict_selected15_row(row, num_samples=NUM_SAMPLES, show_plots=True):
    """Diffusion + forward prediction for one Selected15 row (same pipeline as Section 2)."""
    rid = int(row["SelectedRowID"])
    gid = int(row["IndependentTestGroupID"]) if "IndependentTestGroupID" in row.index else -1
    sid = int(row["SampleIDWithinGroup"]) if "SampleIDWithinGroup" in row.index else -1
    src_gid = int(row["SourceSoftGroupID"]) if "SourceSoftGroupID" in row.index else -1
    hard_set = int(row["HardSetID"]) if "HardSetID" in row.index else -1

    fixed_moduli_raw = row[MOD].to_numpy(dtype=float).reshape(1, 6)
    t_orig = row[["t1", "t2"]].to_numpy(dtype=float).reshape(1, 2)
    raw_target = row[y_cols].to_numpy(dtype=float).reshape(1, 2)

    moduli_tmp = np.zeros((1, 8), dtype=float)
    moduli_tmp[0, :6] = fixed_moduli_raw
    fixed_moduli_scaled = scaler_x.transform(moduli_tmp)[:, :6]
    moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)

    std_target = scaler_y.transform(raw_target)
    target_y = torch.tensor(std_target, dtype=torch.float32, device=device)

    print(f"  SelectedRowID={rid} | IndependentTestGroupID={gid} | SampleIDWithinGroup={sid}")
    print(f"  SourceSoftGroupID={src_gid}, HardSetID={hard_set}")
    print(f"  Fixed moduli: {np.round(fixed_moduli_raw.flatten(), 6)}")
    print(f"  Original t1,t2: {np.round(t_orig.flatten(), 6)}")
    print(f"  Target EpH,EppH: {np.round(raw_target.flatten(), 6)}")

    cands_x_std, preds_std = [], []
    for _ in range(num_samples):
        x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
        cands_x_std.append(x_gen.detach().cpu().numpy().reshape(-1))
        preds_std.append(forward_model(x_gen).detach().cpu().numpy().reshape(-1))

    cands_x_std = np.vstack(cands_x_std)
    preds_std = np.vstack(preds_std)
    cands_x_orig = scaler_x.inverse_transform(cands_x_std)
    preds_orig = scaler_y.inverse_transform(preds_std)

    errors_orig = np.mean((preds_orig - raw_target) ** 2, axis=1)
    best_idx = int(np.argmin(errors_orig))
    best_x = cands_x_orig[best_idx]
    best_y = preds_orig[best_idx]
    best_mse = float(errors_orig[best_idx])

    t1_pred, t2_pred = float(best_x[6]), float(best_x[7])
    eph_pred, epph_pred = float(best_y[0]), float(best_y[1])
    eph_targ, epph_targ = float(raw_target[0, 0]), float(raw_target[0, 1])
    t1_orig, t2_orig = float(t_orig[0, 0]), float(t_orig[0, 1])

    abs_err_eph = abs(eph_pred - eph_targ)
    abs_err_epph = abs(epph_pred - epph_targ)
    rel_err_eph = abs_err_eph / max(abs(eph_targ), REL_EPS)
    rel_err_epph = abs_err_epph / max(abs(epph_targ), REL_EPS)

    print("  Best candidate (no refinement):")
    print(f"    t1_pred,t2_pred = [{t1_pred:.6f}, {t2_pred:.6f}]")
    print(f"    EpH_pred,EppH_pred = [{eph_pred:.6f}, {epph_pred:.6f}]")
    print(f"    MSE(EpH,EppH) = {best_mse:.6e}")
    print(f"    AbsErr EpH={abs_err_eph:.6f}, EppH={abs_err_epph:.6f}")
    print(f"    RelErr EpH={rel_err_eph*100:.3f}%, EppH={rel_err_epph*100:.3f}%")

    if show_plots:
        top_idx = np.argsort(errors_orig)[:50]
        best_xy = np.array([[eph_pred, epph_pred]], dtype=float)
        x_min, x_max, y_min, y_max = adaptive_axis_limits(
            eph_targ, epph_targ, preds_orig[top_idx], best_xy=best_xy, pad_frac=0.45
        )

        plt.figure(figsize=(10, 10))
        plt.scatter(
            preds_orig[top_idx, 0], preds_orig[top_idx, 1],
            c="gray", alpha=0.5, s=50, label="Candidates", zorder=1,
        )
        plt.scatter(
            eph_pred, epph_pred,
            c="blue", s=150, edgecolors="b", label="Best candidate", zorder=3,
        )
        plt.scatter(
            eph_targ, epph_targ,
            c="red", marker="x", s=200,
            label=f"Target ({eph_targ:.3f}, {epph_targ:.3f})", zorder=4,
        )
        plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
        plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)
        plt.xticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.yticks(fontsize=22, fontweight="bold", fontfamily="serif")
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.legend(prop={"size": 18, "weight": "bold"}, loc="upper left", framealpha=0.9, borderpad=0.6)
        ax = plt.gca()
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis="both", which="major", length=8, width=3)
        plt.tight_layout()
        large_path = f"image/selected15_row{rid:02d}_g{gid}_s{sid}_large.png"
        plt.savefig(large_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {large_path}")

        plt.figure(figsize=(6, 6))
        plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c="gray", alpha=0.5, s=200, zorder=1)
        plt.scatter(eph_pred, epph_pred, c="blue", s=200, edgecolors="k", zorder=3)
        plt.scatter(eph_targ, epph_targ, c="red", marker="x", s=200, zorder=4)
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.xticks([])
        plt.yticks([])
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(4.5)
            spine.set_linestyle("--")
        plt.tight_layout()
        small_path = f"image/selected15_row{rid:02d}_g{gid}_s{sid}_small.png"
        plt.savefig(small_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved {small_path}")

    return {
        "SelectedRowID": rid,
        "IndependentTestGroupID": gid,
        "SampleIDWithinGroup": sid,
        "SourceSoftGroupID": src_gid,
        "HardSetID": hard_set,
        "Ep1": float(fixed_moduli_raw[0, 0]),
        "Ep2": float(fixed_moduli_raw[0, 1]),
        "Ep3": float(fixed_moduli_raw[0, 2]),
        "Epp1": float(fixed_moduli_raw[0, 3]),
        "Epp2": float(fixed_moduli_raw[0, 4]),
        "Epp3": float(fixed_moduli_raw[0, 5]),
        "t1_orig": t1_orig,
        "t2_orig": t2_orig,
        "t1_pred": t1_pred,
        "t2_pred": t2_pred,
        "EpH_targ": eph_targ,
        "EppH_targ": epph_targ,
        "EpH_pred": eph_pred,
        "EppH_pred": epph_pred,
        "AbsErr_EpH": abs_err_eph,
        "AbsErr_EppH": abs_err_epph,
        "RelErr_EpH": rel_err_eph,
        "RelErr_EppH": rel_err_epph,
        "RelErr_EpH_pct": rel_err_eph * 100.0,
        "RelErr_EppH_pct": rel_err_epph * 100.0,
        "BestCandidateMSE": best_mse,
        "NumCandidates": num_samples,
    }


def run_selected15_row(row_id, df_all=None, results_list=None, num_samples=NUM_SAMPLES):
    """Predict + plot one Selected15 row. Re-running replaces that row's result."""
    if df_all is None:
        df_all = df_sel15
    if results_list is None:
        results_list = selected15_results

    rdf = df_all.loc[df_all["SelectedRowID"] == row_id].reset_index(drop=True)
    if len(rdf) != 1:
        raise ValueError(f"Expected 1 row for SelectedRowID={row_id}, got {len(rdf)}")

    print("=" * 70)
    print(f"SelectedRowID = {row_id}")
    print("=" * 70)

    keep = [r for r in results_list if int(r["SelectedRowID"]) != int(row_id)]
    results_list.clear()
    results_list.extend(keep)

    row = rdf.iloc[0]
    res = predict_selected15_row(row, num_samples=num_samples, show_plots=True)
    results_list.append(res)
    print(f"\n[SelectedRow {row_id}] done. selected15_results has {len(results_list)} rows.")
    print("-" * 60)
    return results_list


print("Selected15 helpers ready: predict_selected15_row(), run_selected15_row()")
print(f"Output Excel will be: {SELECTED15_OUT}")


#### 6.2 SelectedRowID = 1

Run **only this cell** to (re)generate prediction and figures for Selected15 row 1.


In [ ]:
# Predict & plot Selected15 row 1
run_selected15_row(1)


#### 6.3 SelectedRowID = 2

Run **only this cell** to (re)generate prediction and figures for Selected15 row 2.


In [ ]:
# Predict & plot Selected15 row 2
run_selected15_row(2)


#### 6.4 SelectedRowID = 3

Run **only this cell** to (re)generate prediction and figures for Selected15 row 3.


In [ ]:
# Predict & plot Selected15 row 3
run_selected15_row(3)


#### 6.5 SelectedRowID = 4

Run **only this cell** to (re)generate prediction and figures for Selected15 row 4.


In [ ]:
# Predict & plot Selected15 row 4
run_selected15_row(4)


#### 6.6 SelectedRowID = 5

Run **only this cell** to (re)generate prediction and figures for Selected15 row 5.


In [ ]:
# Predict & plot Selected15 row 5
run_selected15_row(5)


#### 6.7 SelectedRowID = 6

Run **only this cell** to (re)generate prediction and figures for Selected15 row 6.


In [ ]:
# Predict & plot Selected15 row 6
run_selected15_row(6)


#### 6.8 SelectedRowID = 7

Run **only this cell** to (re)generate prediction and figures for Selected15 row 7.


In [ ]:
# Predict & plot Selected15 row 7
run_selected15_row(7)


#### 6.9 SelectedRowID = 8

Run **only this cell** to (re)generate prediction and figures for Selected15 row 8.


In [ ]:
# Predict & plot Selected15 row 8
run_selected15_row(8)


#### 6.10 SelectedRowID = 9

Run **only this cell** to (re)generate prediction and figures for Selected15 row 9.


In [ ]:
# Predict & plot Selected15 row 9
run_selected15_row(9)


#### 6.11 SelectedRowID = 10

Run **only this cell** to (re)generate prediction and figures for Selected15 row 10.


In [ ]:
# Predict & plot Selected15 row 10
run_selected15_row(10)


#### 6.12 SelectedRowID = 11

Run **only this cell** to (re)generate prediction and figures for Selected15 row 11.


In [ ]:
# Predict & plot Selected15 row 11
run_selected15_row(11)


#### 6.13 SelectedRowID = 12

Run **only this cell** to (re)generate prediction and figures for Selected15 row 12.


In [ ]:
# Predict & plot Selected15 row 12
run_selected15_row(12)


#### 6.14 SelectedRowID = 13

Run **only this cell** to (re)generate prediction and figures for Selected15 row 13.


In [ ]:
# Predict & plot Selected15 row 13
run_selected15_row(13)


#### 6.15 SelectedRowID = 14

Run **only this cell** to (re)generate prediction and figures for Selected15 row 14.


In [ ]:
# Predict & plot Selected15 row 14
run_selected15_row(14)


#### 6.16 SelectedRowID = 15

Run **only this cell** to (re)generate prediction and figures for Selected15 row 15.


In [ ]:
# Predict & plot Selected15 row 15
run_selected15_row(15)


#### 6.17 Save Selected15 comparison Excel

Run after the row cells you need. Writes `data/independent_hard_selected15_prediction_soft_hard.xlsx`.


In [ ]:
selected15_df = pd.DataFrame(selected15_results).sort_values(
    ["SelectedRowID"]
).reset_index(drop=True)

selected15_df.to_excel(SELECTED15_OUT, index=False)

print(f"Results saved to {SELECTED15_OUT}")
print(f"Total rows: {len(selected15_df)}")
print("\nSelected15 prediction comparison:")
print(selected15_df[[
    "SelectedRowID",
    "IndependentTestGroupID", "SampleIDWithinGroup",
    "t1_orig", "t1_pred", "t2_orig", "t2_pred",
    "EpH_targ", "EpH_pred", "EppH_targ", "EppH_pred",
    "AbsErr_EpH", "RelErr_EpH_pct", "AbsErr_EppH", "RelErr_EppH_pct",
]].round(4).to_string(index=False))

print("\n" + "=" * 70)
print("Selected15 independent hard-material prediction summary")
print("=" * 70)
print(f"Input:  {SELECTED15_PATH} [{SELECTED15_SHEET}]")
print(f"Output: {SELECTED15_OUT}")
print(f"Diffusion / Forward / Scalers: soft+hard (loaded in Section 1)")
print(f"Candidates per row: {NUM_SAMPLES}")
if len(selected15_df):
    print(f"Mean AbsErr EpH:  {selected15_df['AbsErr_EpH'].mean():.6f}")
    print(f"Mean AbsErr EppH: {selected15_df['AbsErr_EppH'].mean():.6f}")
    print(f"Mean RelErr EpH:  {selected15_df['RelErr_EpH_pct'].mean():.3f}%")
    print(f"Mean RelErr EppH: {selected15_df['RelErr_EppH_pct'].mean():.3f}%")
print("=" * 70)
